# FewShotNAS on Google Colab

This notebook runs resumable CrossMod-CAN architecture search on a deterministic 70-training/17-validation BioVid subject split. Validation is matched: zero-shot uses the source prototype bank, while 10-shot draws support from each validation subject's `Train` partition and always scores the same `Test` queries. The validation result is tuning-only.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys
REPO_URL = 'https://github.com/hhihn/FewShotPainAdaptation.git'
PROJECT_DIR = Path('/content/FewShotPainAdaptation')
BRANCH_NAME = 'painnas'
if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only
%cd $PROJECT_DIR
sys.path.insert(0, str(PROJECT_DIR)) if str(PROJECT_DIR) not in sys.path else None
assert (PROJECT_DIR / 'fewshotnas').is_dir()


## 2. Install pinned dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/fewshotnas/requirements-colab.txt


## 3. Stage BioVid on the local Colab SSD

In [ ]:
from data_loaders.dataset_staging import stage_predefined_dataset_from_archive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/PainData')
LOCAL_DATA_DIR = Path('/content/PainData')
BIOVID_ROOT = stage_predefined_dataset_from_archive(
    'biovid_part_a', drive_data_dir=DRIVE_DATA_DIR, local_data_dir=LOCAL_DATA_DIR,
    local_archive_dir=Path('/content'),
)
DATA_DIR = LOCAL_DATA_DIR
print('BioVid root:', BIOVID_ROOT)


## 4. Verify GPU and configure reproducibility

In [ ]:
import random, numpy as np, tensorflow as tf
from tensorflow.keras import mixed_precision
GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Select a Colab GPU runtime.'
for gpu in GPUS:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
mixed_precision.set_global_policy('mixed_float16')
SEED = 42
tf.keras.utils.set_random_seed(SEED); random.seed(SEED); np.random.seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', GPUS)


## 5. Configure the thorough search

In [ ]:
from fewshotnas.config import FewShotNASConfig
RUN_NAME = 'run_001'
OUTPUT_DIR = Path('/content/drive/MyDrive/FewShotNAS') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = True
CONFIG = FewShotNASConfig(seed=SEED, n_trials=100, max_epochs=5, tasks_per_epoch=10_000, support_repeats=100)
print(CONFIG)
print('Output:', OUTPUT_DIR)


## 6. Load data and audit the deterministic subject split

In [ ]:
import pandas as pd
from fewshotnas.search import _load_dataset
from fewshotnas.data import deterministic_subject_split
DATASET = _load_dataset(str(DATA_DIR), CONFIG)
SUBJECT_SPLIT = deterministic_subject_split(DATASET.unique_subjects, train_count=70, validation_count=17, seed=SEED)
assert not set(SUBJECT_SPLIT.train_subjects) & set(SUBJECT_SPLIT.validation_subjects)
display(pd.DataFrame([{'samples': len(DATASET.y), 'subjects': len(DATASET.unique_subjects), 'train_subjects': 70, 'validation_subjects': 17, 'sequence_length': DATASET.X.shape[1], 'modalities': DATASET.X.shape[2]}]))
print('Train IDs:', SUBJECT_SPLIT.train_subjects)
print('Validation IDs:', SUBJECT_SPLIT.validation_subjects)
del DATASET


## 7. Run or resume NAS and the fresh winner refit

In [ ]:
from fewshotnas.search import run_all
RESULT = run_all(str(DATA_DIR), CONFIG, OUTPUT_DIR, resume=RESUME)
display(pd.DataFrame([RESULT['refit']['metrics']]))
print('Artifacts:', OUTPUT_DIR)


## 8. Inspect trial progress and the selected architecture

In [ ]:
import json, matplotlib.pyplot as plt
trials = pd.read_csv(OUTPUT_DIR / 'search/trials.csv')
display(trials.sort_values('value', ascending=False).head(20))
best = json.loads((OUTPUT_DIR / 'search/best_architecture.json').read_text())
print(json.dumps(best, indent=2))
completed = trials[trials.state == 'COMPLETE']
if not completed.empty:
    completed.plot(x='trial_number', y='value', marker='o', title='FewShotNAS validation objective'); plt.ylim(0, 1); plt.show()


## 9. Audit matched validation and summarize the fresh refit

In [ ]:
zero = pd.read_csv(OUTPUT_DIR / 'refit/zero_shot_subject_metrics.csv')
kshot = pd.read_csv(OUTPUT_DIR / 'refit/k_shot_repeat_metrics.csv')
assert zero.subject.nunique() == 17 and kshot.subject.nunique() == 17
assert kshot.groupby('subject').size().eq(CONFIG.support_repeats).all()
summary = pd.DataFrame({'zero_shot': zero.groupby('subject').accuracy.mean(), 'k_shot': kshot.groupby('subject').accuracy.mean()})
display(summary); display(summary.mean().to_frame('mean_accuracy'))
summary.plot(kind='bar', figsize=(13, 4), title='Tuning-only matched validation by subject'); plt.ylim(0, 1); plt.show()


## 10. Optional runtime cleanup

In [ ]:
import gc
tf.keras.backend.clear_session(); gc.collect()
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print('Cleanup complete')
